# RT-DETR-L — Training & Evaluation
**Dataset:** CARLA-GeAR billboard scenarios (9 × 50 test images)
**Model:** RT-DETR-L (HGNetV2, 32.8M params)
**Attack:** Patch trained on YOLOv8n, transferred to RT-DETR

In [ ]:
# ── CELL 1: Mount Drive & GPU check ──────────────────────────
from google.colab import drive
import torch

drive.mount('/content/drive')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
# ── CELL 2: Install ──────────────────────────────────────────
!pip install ultralytics --quiet
import ultralytics
print('Ultralytics:', ultralytics.__version__)

## Training
Skip this section if you already have `best.pt` saved to Drive.

In [ ]:
# ── CELL 3: Write data.yaml ──────────────────────────────────
import yaml, os

DATASET_ROOT = '/content/drive/MyDrive/2dod/2dod'
DATASET_YAML = f'{DATASET_ROOT}/data.yaml'

with open(DATASET_YAML, 'w') as f:
    yaml.dump({
        'path':  DATASET_ROOT,
        'train': 'train.txt',
        'val':   'val.txt',
        'nc':    6,
        'names': ['person','car','motorcycle','truck','traffic light','stop sign'],
    }, f, default_flow_style=False)

print('data.yaml written:')
with open(DATASET_YAML) as f:
    print(f.read())

In [ ]:
# ── CELL 4: Train RT-DETR-L ──────────────────────────────────
# Skip if best.pt already exists on Drive
from ultralytics import RTDETR

model = RTDETR('rtdetr-l.pt')

results = model.train(
    data         = DATASET_YAML,
    epochs       = 200,
    imgsz        = 640,
    batch        = 16,           # reduce to 8 if OOM
    device       = 0,
    project      = '/content/drive/MyDrive/runs',
    name         = 'rtdetr_carlagear',
    exist_ok     = False,
    lr0          = 0.0001,
    weight_decay = 0.0001,
    mosaic       = 0.0,
    mixup        = 0.0,
    val          = True,
    save         = True,
    save_period  = 10,
    patience     = 50,
    workers      = 2,
    seed         = 0,
    verbose      = True,
)

## Evaluation
Runs clean vs patched per billboard scenario — produces the same metrics as the NanoDet/YOLOv8 evaluation.

In [ ]:
# ── CELL 5: Build test txt files (run once) ──────────────────
import os

DATASET_ROOT   = '/content/drive/MyDrive/2dod/2dod'
PATCHED_PARENT = '/content/drive/MyDrive/Patched Datasets'

for split in ['test_nopatch', 'test_random']:
    lines = []
    for i in range(1, 10):
        img_dir = f'{DATASET_ROOT}/billboard0{i}/images/{split}'
        if os.path.exists(img_dir):
            for fn in sorted(os.listdir(img_dir)):
                if fn.endswith('.png') or fn.endswith('.jpg'):
                    lines.append(f'{img_dir}/{fn}')
    txt = f'{DATASET_ROOT}/{split}.txt'
    with open(txt, 'w') as f:
        f.write('\n'.join(lines))
    print(f'{split}.txt: {len(lines)} images')

# Patched images are in Patched Datasets/patched_dataset{i}/images/test
lines = []
for i in range(1, 10):
    img_dir = f'{PATCHED_PARENT}/patched_dataset{i}/images/test'
    if os.path.exists(img_dir):
        for fn in sorted(os.listdir(img_dir)):
            if fn.endswith('.png') or fn.endswith('.jpg'):
                lines.append(f'{img_dir}/{fn}')
txt = f'{DATASET_ROOT}/test_optimized.txt'
with open(txt, 'w') as f:
    f.write('\n'.join(lines))
print(f'test_optimized.txt: {len(lines)} images')

In [ ]:
# ── CELL 6: Per-scenario evaluation ──────────────────────────
import json, csv, yaml
import numpy as np
from pathlib import Path
from ultralytics import RTDETR

BEST_WEIGHTS   = '/content/drive/MyDrive/runs/rtdetr_carlagear2/weights/best.pt'
DATASET_ROOT   = '/content/drive/MyDrive/2dod/2dod'
PATCHED_PARENT = '/content/drive/MyDrive/Patched Datasets'
CLEAN_ROOT     = f'{PATCHED_PARENT}/Clean_640'
OUTPUT_DIR     = '/content/drive/MyDrive/rtdetr_eval'
IMGSZ          = 640
CONF_MIN       = 0.001
IOU            = 0.6
THRESHOLDS     = [0.3, 0.5, 0.7]
GT_IOU_MATCH   = 0.5
NAMES          = ['person','car','motorcycle','truck','traffic light','stop sign']
IMG_EXTS       = {'.jpg','.jpeg','.png','.bmp'}

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
model = RTDETR(BEST_WEIGHTS)
print('Model loaded:', BEST_WEIGHTS)

# ── Helper functions ────────────────────────────────────────
def read_yolo_gt(txt_path):
    if not Path(txt_path).exists(): return []
    out = []
    for ln in Path(txt_path).read_text(errors='ignore').strip().splitlines():
        parts = ln.strip().split()
        if len(parts) < 5: continue
        c = int(float(parts[0]))
        x, y, w, h = map(float, parts[1:5])
        out.append((c, x, y, w, h))
    return out

def xywhn_to_xyxy(x, y, w, h):
    return x-w/2, y-h/2, x+w/2, y+h/2

def iou_xyxy(a, b):
    ix1, iy1 = max(a[0],b[0]), max(a[1],b[1])
    ix2, iy2 = min(a[2],b[2]), min(a[3],b[3])
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    area_a = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    area_b = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    union = area_a + area_b - inter
    return inter/union if union > 0 else 0.0

def evaluate_folder(img_dir, lab_dir):
    img_dir, lab_dir = Path(img_dir), Path(lab_dir)
    img_files = sorted([f for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXTS])
    if not img_files: return None
    det_by_thr = {t: [] for t in [CONF_MIN]+THRESHOLDS}
    fp_by_thr  = {t: [] for t in [CONF_MIN]+THRESHOLDS}
    for r in model.predict(source=str(img_dir), imgsz=IMGSZ, conf=CONF_MIN,
                           iou=IOU, device=0, stream=True, verbose=False):
        stem = Path(r.path).stem
        gt = read_yolo_gt(lab_dir / f'{stem}.txt')
        boxes = r.boxes
        if boxes is None or len(boxes) == 0:
            for t in det_by_thr: det_by_thr[t].append(0); fp_by_thr[t].append(0)
            continue
        pred_cls   = boxes.cls.detach().cpu().numpy().astype(int)
        pred_xywhn = boxes.xywhn.detach().cpu().numpy().astype(float)
        pred_conf  = boxes.conf.detach().cpu().numpy().astype(float)
        preds = [(int(pred_cls[i]), xywhn_to_xyxy(*pred_xywhn[i].tolist()), float(pred_conf[i]))
                 for i in range(len(pred_cls))]
        gt_by_class = {}
        for (c,x,y,w,h) in gt: gt_by_class.setdefault(int(c),[]).append(xywhn_to_xyxy(x,y,w,h))
        for t in det_by_thr:
            pf = sorted([(c,box,conf) for (c,box,conf) in preds if conf>=t], key=lambda z:-z[2])
            det_by_thr[t].append(len(pf))
            remaining = {c: list(lst) for c,lst in gt_by_class.items()}
            fp = 0
            for c, pbox, _ in pf:
                best_iou, best_j = 0.0, None
                for j, gbox in enumerate(remaining.get(c,[])):
                    v = iou_xyxy(pbox, gbox)
                    if v > best_iou: best_iou, best_j = v, j
                if best_j is not None and best_iou >= GT_IOU_MATCH:
                    remaining[c].pop(best_j)
                else: fp += 1
            fp_by_thr[t].append(fp)
    return {
        'det_per_img': {str(t): float(np.mean(det_by_thr[t])) for t in det_by_thr},
        'fp_per_img':  {str(t): float(np.mean(fp_by_thr[t]))  for t in fp_by_thr},
    }

# ── Per-scenario loop ──────────────────────────────────────
BILLBOARDS = [f'billboard0{i}' for i in range(1,10)]
all_results = {}

for i, bb in enumerate(BILLBOARDS, 1):
    print(f'\n── {bb} ──')
    clean_img = f'{CLEAN_ROOT}/images/billboard0{i}'
    clean_lab = f'{CLEAN_ROOT}/labels/billboard0{i}'
    patch_img = f'{PATCHED_PARENT}/patched_dataset{i}/images/test'
    patch_lab = clean_lab

    clean_yaml = f'{OUTPUT_DIR}/{bb}_clean.yaml'
    patch_yaml = f'{OUTPUT_DIR}/{bb}_patch.yaml'
    for yaml_path, img_rel, root in [
        (clean_yaml, f'images/billboard0{i}', CLEAN_ROOT),
        (patch_yaml, 'images/test', f'{PATCHED_PARENT}/patched_dataset{i}'),
    ]:
        Path(yaml_path).write_text(
            f'path: {root}\ntrain: {img_rel}\nval: {img_rel}\n'
            f'nc: {len(NAMES)}\nnames: {json.dumps(NAMES)}\n'
        )

    c_val = model.val(data=clean_yaml, imgsz=IMGSZ, batch=16, device=0,
                      iou=IOU, conf=CONF_MIN, verbose=False)
    p_val = model.val(data=patch_yaml, imgsz=IMGSZ, batch=16, device=0,
                      iou=IOU, conf=CONF_MIN, verbose=False)

    print('  Running density metrics...')
    c_dens = evaluate_folder(clean_img, clean_lab)
    p_dens = evaluate_folder(patch_img, patch_lab)

    all_results[bb] = {
        'clean':   {'mAP50': c_val.box.map50, 'mAP50-95': c_val.box.map,
                    'precision': c_val.box.mp, 'recall': c_val.box.mr, **(c_dens or {})},
        'patched': {'mAP50': p_val.box.map50, 'mAP50-95': p_val.box.map,
                    'precision': p_val.box.mp, 'recall': p_val.box.mr, **(p_dens or {})},
        'delta':   {
            'mAP50':    round(p_val.box.map50 - c_val.box.map50, 4),
            'mAP50-95': round(p_val.box.map   - c_val.box.map,   4),
            'det_per_img': {str(t): round(
                (p_dens or {}).get('det_per_img',{}).get(str(t),0) -
                (c_dens or {}).get('det_per_img',{}).get(str(t),0), 3)
                for t in [CONF_MIN]+THRESHOLDS},
            'fp_per_img': {str(t): round(
                (p_dens or {}).get('fp_per_img',{}).get(str(t),0) -
                (c_dens or {}).get('fp_per_img',{}).get(str(t),0), 3)
                for t in [CONF_MIN]+THRESHOLDS},
        }
    }
    r = all_results[bb]
    print(f"  Clean:   mAP50={r['clean']['mAP50']:.4f}  "
          f"Det@0.001={r['clean'].get('det_per_img',{}).get(str(CONF_MIN),0):.1f}  "
          f"FP@0.5={r['clean'].get('fp_per_img',{}).get('0.5',0):.2f}")
    print(f"  Patched: mAP50={r['patched']['mAP50']:.4f}  "
          f"Det@0.001={r['patched'].get('det_per_img',{}).get(str(CONF_MIN),0):.1f}  "
          f"FP@0.5={r['patched'].get('fp_per_img',{}).get('0.5',0):.2f}")
    print(f"  Δ mAP50={r['delta']['mAP50']:+.4f}")

In [ ]:
# ── FAST CELL: Only count detections @ conf=0.001 with larger max_det ───────
import numpy as np
from pathlib import Path

MAX_DET = 1000   # increase if still saturated
CONF_CHECK = 0.001

def count_detections_only(img_dir, max_det=1000, conf=0.001):
    img_dir = Path(img_dir)
    img_files = sorted([f for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXTS])
    if not img_files:
        return None

    det_counts = []

    for r in model.predict(
        source=str(img_dir),
        imgsz=IMGSZ,
        conf=conf,
        iou=IOU,
        device=0,
        stream=True,
        verbose=False,
        max_det=max_det
    ):
        boxes = r.boxes
        n = 0 if boxes is None else len(boxes)
        det_counts.append(int(n))

    return {
        'mean_det_per_img': float(np.mean(det_counts)),
        'min_det': int(np.min(det_counts)),
        'max_det_seen': int(np.max(det_counts)),
        'sat_images': int(sum(1 for x in det_counts if x >= max_det)),
        'num_images': len(det_counts),
        'all_counts': det_counts,
    }

print(f'Running FAST detection-count check with conf={CONF_CHECK}, max_det={MAX_DET} ...')

for i, bb in enumerate(BILLBOARDS, 1):
    print(f'\n── {bb} ──')

    clean_img = f'{CLEAN_ROOT}/images/billboard0{i}'
    patch_img = f'{PATCHED_PARENT}/patched_dataset{i}/images/test'

    clean_res = count_detections_only(clean_img, max_det=MAX_DET, conf=CONF_CHECK)
    patch_res = count_detections_only(patch_img, max_det=MAX_DET, conf=CONF_CHECK)

    if clean_res is None or patch_res is None:
        print('  Skipped (missing images)')
        continue

    print(
        f"  Clean:   mean={clean_res['mean_det_per_img']:.1f}  "
        f"min={clean_res['min_det']}  max={clean_res['max_det_seen']}  "
        f"sat={clean_res['sat_images']}/{clean_res['num_images']}"
    )
    print(
        f"  Patched: mean={patch_res['mean_det_per_img']:.1f}  "
        f"min={patch_res['min_det']}  max={patch_res['max_det_seen']}  "
        f"sat={patch_res['sat_images']}/{patch_res['num_images']}"
    )
    print(
        f"  Δ mean det/image = "
        f"{patch_res['mean_det_per_img'] - clean_res['mean_det_per_img']:+.1f}"
    )

### Note on Det@0.001

RT-DETR returns a fixed 300 predictions per image regardless of confidence threshold, so Det@0.001 is always 300 for both clean and adversarial inputs. This is a model-level architectural property, not an evaluation bug.

In [ ]:
# ── DIAGNOSTIC: inspect RT-DETR query count / decoder config ─────────────────
import pprint

print("Top-level model type:", type(model.model))

# 1) print YAML/config if available
print("\n=== model.model.yaml ===")
if hasattr(model.model, "yaml"):
    pprint.pprint(model.model.yaml)
else:
    print("No model.model.yaml found")

# 2) inspect all modules for query-related attributes
print("\n=== modules with query-related attributes ===")
found = []
for name, m in model.model.named_modules():
    attrs = {}
    for attr in ["nq", "num_queries", "query_dim", "max_det", "nc"]:
        if hasattr(m, attr):
            attrs[attr] = getattr(m, attr)
    if attrs:
        found.append((name, type(m).__name__, attrs))

if found:
    for name, cls, attrs in found:
        print(f"{name or '[root]'}  |  {cls}  |  {attrs}")
else:
    print("No query-related attrs found")

# 3) inspect final module
print("\n=== last module ===")
mods = list(model.model.named_modules())
last_name, last_mod = mods[-1]
print("last module name:", last_name)
print("last module type:", type(last_mod))
print("dir(last module) query-ish:",
      [x for x in dir(last_mod) if any(k in x.lower() for k in ["query", "nq", "max_det"])])

for attr in ["nq", "num_queries", "max_det"]:
    if hasattr(last_mod, attr):
        print(f"last_mod.{attr} =", getattr(last_mod, attr))

In [ ]:
# ── CELL 7: Summary table & save ─────────────────────────────
import csv

clean_maps = [all_results[b]['clean']['mAP50'] for b in BILLBOARDS]
patch_maps = [all_results[b]['patched']['mAP50'] for b in BILLBOARDS]
deltas     = [all_results[b]['delta']['mAP50'] for b in BILLBOARDS]
ALL_THRS   = [CONF_MIN] + THRESHOLDS

print(f'\n{"="*90}')
print(f'{"BB":<14} {"C-mAP":>7} {"P-mAP":>7} {"Δ-mAP":>7}', end='')
for t in ALL_THRS:
    print(f'  {"DetC@"+str(t):>10} {"DetP@"+str(t):>10} {"FPC@"+str(t):>9} {"FPP@"+str(t):>9}', end='')
print()
print('-'*90)
for bb in BILLBOARDS:
    r = all_results[bb]
    print(f'{bb:<14} {r["clean"]["mAP50"]:>7.3f} {r["patched"]["mAP50"]:>7.3f} '
          f'{r["delta"]["mAP50"]:>+7.3f}', end='')
    for t in ALL_THRS:
        cd = r['clean'].get('det_per_img',{}).get(str(t),0)
        pd = r['patched'].get('det_per_img',{}).get(str(t),0)
        cf = r['clean'].get('fp_per_img',{}).get(str(t),0)
        pf = r['patched'].get('fp_per_img',{}).get(str(t),0)
        print(f'  {cd:>10.2f} {pd:>10.2f} {cf:>9.2f} {pf:>9.2f}', end='')
    print()
print('-'*90)
print(f'{"Mean":<14} {np.mean(clean_maps):>7.3f} {np.mean(patch_maps):>7.3f} '
      f'{np.mean(deltas):>+7.3f}±{np.std(deltas):.3f}')

# Save JSON and CSV
with open(f'{OUTPUT_DIR}/rtdetr_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

csv_path = f'{OUTPUT_DIR}/rtdetr_results.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    header = ['scenario','clean_mAP50','patched_mAP50','delta_mAP50',
              'clean_mAP95','patched_mAP95','delta_mAP95']
    for t in ALL_THRS:
        header += [f'clean_det_{t}',f'patched_det_{t}',f'delta_det_{t}',
                   f'clean_fp_{t}', f'patched_fp_{t}', f'delta_fp_{t}']
    writer.writerow(header)
    for bb in BILLBOARDS:
        r = all_results[bb]
        row = [bb, r['clean']['mAP50'], r['patched']['mAP50'], r['delta']['mAP50'],
               r['clean']['mAP50-95'], r['patched']['mAP50-95'], r['delta']['mAP50-95']]
        for t in ALL_THRS:
            cd = r['clean'].get('det_per_img',{}).get(str(t),0)
            pd = r['patched'].get('det_per_img',{}).get(str(t),0)
            dd = r['delta']['det_per_img'].get(str(t),0)
            cf = r['clean'].get('fp_per_img',{}).get(str(t),0)
            pf = r['patched'].get('fp_per_img',{}).get(str(t),0)
            df = r['delta']['fp_per_img'].get(str(t),0)
            row += [cd,pd,dd,cf,pf,df]
        writer.writerow(row)

print(f'\nSaved JSON: {OUTPUT_DIR}/rtdetr_results.json')
print(f'Saved CSV:  {csv_path}')
print('Done ✅')